# SIM 800s — Localization Analysis

**Localization point**: first local minimum of `shear_mean` (from TP2 data).

Plots:
1. Factor `f` (tension/compression efficiency ratio) vs compression ratio — all sims, localization marked
2. `shear_mean` vs compression ratio — all sims, localization marked
3. Pressure vs `f` at localization
4. Pressure vs displacement at localization
5. `shear_mean` vs `f` with localization point

In [ ]:
import sys, os, pickle, json
import numpy as np
import matplotlib.pyplot as plt
from scipy.signal import argrelmin

sys.path.insert(0, os.path.abspath('..'))

RESULTS_DIR = '../I001_Results'
OBJ_DIR     = '../I001_Results/OBJ_files'
SIMS        = list(range(800, 810))
I3_EXT      = 'I3_BFS_3002'


In [ ]:
def load_pkl(path):
    """Return pickle contents or None if file missing."""
    if not os.path.exists(path):
        return None
    with open(path, 'rb') as f:
        return pickle.load(f)

def get_pressure(sim):
    """Read Pressure_BC from the first step that has one."""
    path = os.path.join(OBJ_DIR, f'SIM_{sim:03d}.json')
    if not os.path.exists(path):
        return None
    with open(path) as f:
        d = json.load(f)
    for step in d.get('steps', []):
        pb = step.get('Pressure_BC', 0)
        if pb:
            return float(pb)
    return 0.0

def first_local_min(arr):
    """Return index of first local minimum, or None."""
    arr = np.array(arr)
    minima = argrelmin(arr, order=3)[0]
    return int(minima[0]) if len(minima) > 0 else None


In [ ]:
# ── Load all data ──────────────────────────────────────────────────────────────
data = {}

for sim in SIMS:
    i3  = load_pkl(os.path.join(RESULTS_DIR, f'DATA_PICK_{sim:03d}_{I3_EXT}.pkl'))
    tp2 = load_pkl(os.path.join(RESULTS_DIR, f'DATA_PICK_{sim:03d}_TP2.pkl'))
    a2  = load_pkl(os.path.join(RESULTS_DIR, f'DATA_PICK_{sim:03d}_A2.pkl'))
    p   = get_pressure(sim)

    if i3 is None and tp2 is None:
        print(f'SIM_{sim}: no I3 or TP2 data — skipping')
        continue

    entry = {'pressure': p, 'i3': i3, 'tp2': tp2, 'a2': a2}

    # Compression ratio and f from I3
    if i3 is not None:
        t  = np.array(i3['t'])
        ef_t = np.array(i3['global_ef_t'], dtype=float)
        ef_c = np.array(i3['global_ef_c'], dtype=float)
        with np.errstate(divide='ignore', invalid='ignore'):
            f_arr = np.where(ef_c != 0, ef_t / ef_c, np.nan)
        entry['t']   = t
        entry['f']   = f_arr

    # shear_mean and localization from TP2
    if tp2 is not None:
        sm = np.array(tp2['shear_mean'])
        t_tp2 = np.array(tp2['t'])
        loc_idx = first_local_min(sm)
        entry['shear_mean'] = sm
        entry['t_tp2']      = t_tp2
        entry['loc_idx']    = loc_idx
        if loc_idx is not None:
            entry['loc_t']       = t_tp2[loc_idx]
            entry['loc_shear']   = sm[loc_idx]
            entry['loc_displ']   = -a2['U2']['PERN-9999997'][loc_idx] if a2 else None
            # f at localization — use same index if I3 shares t-axis
            if i3 is not None:
                entry['loc_f'] = f_arr[loc_idx]
        print(f'SIM_{sim}: P={p} MPa | loc_idx={loc_idx}'
              + (f' | loc_t={entry["loc_t"]:.3f} | loc_shear={entry["loc_shear"]:.4f}'
                 if loc_idx is not None else ' | no local min found'))
    else:
        print(f'SIM_{sim}: P={p} MPa | no TP2')

    data[sim] = entry

print(f'\nLoaded {len(data)} simulations.')


## Plot 1 — Factor `f` vs compression ratio

In [ ]:
fig, ax = plt.subplots(figsize=(9, 5))

for sim, d in data.items():
    if d.get('i3') is None:
        continue
    p   = d['pressure']
    t   = d['t']
    f   = d['f']
    label = f'SIM_{sim}  P={p} MPa'
    line, = ax.plot(t[1:], f[1:], label=label)

    # localization marker
    if d.get('loc_idx') is not None and 'loc_f' in d:
        ax.plot(d['loc_t'], d['loc_f'], 'o', color=line.get_color(),
                markersize=8, markeredgecolor='k', zorder=5)

ax.set_xlabel('Compression Ratio')
ax.set_ylabel('f  (tension / compression efficiency)')
ax.set_title('Factor f vs compression ratio — SIM 800s')
ax.set_yscale('log')
ax.legend(loc='best', fontsize=8)
ax.grid(True)
plt.tight_layout()
plt.show()


## Plot 2 — Shear mean (TP) vs compression ratio

In [ ]:
fig, ax = plt.subplots(figsize=(9, 5))

for sim, d in data.items():
    if d.get('tp2') is None:
        continue
    p    = d['pressure']
    t    = d['t_tp2']
    sm   = d['shear_mean']
    label = f'SIM_{sim}  P={p} MPa'
    line, = ax.plot(t[1:], sm[1:], label=label)

    # localization marker
    if d.get('loc_idx') is not None:
        ax.plot(d['loc_t'], d['loc_shear'], 'o', color=line.get_color(),
                markersize=8, markeredgecolor='k', zorder=5,
                label=f'  → loc SIM_{sim}')

ax.set_xlabel('Compression Ratio')
ax.set_ylabel('Shear mean (TP)')
ax.set_title('Shear mean vs compression ratio — SIM 800s')
ax.legend(loc='best', fontsize=8)
ax.grid(True)
plt.tight_layout()
plt.show()


## Plot 3 — Pressure vs `f` at localization

In [ ]:
fig, ax = plt.subplots(figsize=(6, 5))

for sim, d in data.items():
    if d.get('loc_idx') is None or 'loc_f' not in d:
        continue
    ax.scatter(d['pressure'], d['loc_f'], s=80, zorder=5)
    ax.annotate(f'SIM_{sim}', (d['pressure'], d['loc_f']),
                textcoords='offset points', xytext=(6, 4), fontsize=8)

ax.set_xlabel('Internal pressure [MPa]')
ax.set_ylabel('f at localization')
ax.set_title('Pressure vs f at localization — SIM 800s')
ax.grid(True)
plt.tight_layout()
plt.show()


## Plot 4 — Pressure vs displacement at localization

In [ ]:
fig, ax = plt.subplots(figsize=(6, 5))

for sim, d in data.items():
    if d.get('loc_idx') is None or d.get('loc_displ') is None:
        continue
    ax.scatter(d['pressure'], d['loc_displ'], s=80, zorder=5)
    ax.annotate(f'SIM_{sim}', (d['pressure'], d['loc_displ']),
                textcoords='offset points', xytext=(6, 4), fontsize=8)

ax.set_xlabel('Internal pressure [MPa]')
ax.set_ylabel('Displacement at localization [mm]')
ax.set_title('Pressure vs displacement at localization — SIM 800s')
ax.grid(True)
plt.tight_layout()
plt.show()


## Plot 5 — Shear mean vs `f` (with localization point)

In [ ]:
fig, ax = plt.subplots(figsize=(7, 5))

for sim, d in data.items():
    if d.get('tp2') is None or d.get('i3') is None:
        continue
    p  = d['pressure']
    sm = d['shear_mean']
    f  = d['f']
    # both arrays are 201 long; skip index 0 (t=0, f undefined)
    label = f'SIM_{sim}  P={p} MPa'
    line, = ax.plot(sm[1:], f[1:], label=label)

    if d.get('loc_idx') is not None and 'loc_f' in d:
        ax.plot(d['loc_shear'], d['loc_f'], 'o', color=line.get_color(),
                markersize=8, markeredgecolor='k', zorder=5)

ax.set_xlabel('Shear mean (TP)')
ax.set_ylabel('f  (tension / compression efficiency)')
ax.set_title('Shear mean vs f — SIM 800s  (● = localization)')
ax.set_yscale('log')
ax.legend(loc='best', fontsize=8)
ax.grid(True)
plt.tight_layout()
plt.show()
